# Lesson 4.4 — Should I Fine-Tune My Embedding Model?

**Companion notebook for the blog post.**

You've picked an embedding model. Results are... okay. Users search for `"446(b) safe harbor correction"` and get generic tax compliance paragraphs instead of the specific IRS procedure they need.

This notebook will help you:
1. **Decide** whether fine-tuning is actually the right fix
2. **Measure** whether your current model already handles your domain
3. **Build** training data (real, synthetic, and hard negatives)
4. **Fine-tune** an embedding model with `sentence-transformers`
5. **Evaluate** whether it actually got better — and didn't break anything

---
## 📦 Setup

In [ ]:
# Uncomment and run once
# !pip install sentence-transformers rank-bm25 scikit-learn torch

In [ ]:
import os
os.environ["USE_TF"] = "0"

import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("✅ Imports ready")

---
---
## Part 1: Should You Even Fine-Tune?

Fine-tuning is **not** the first lever you should pull. Rule out these cheaper fixes first:

| Problem | Fix |
|---|---|
| Chunks split key information across pieces | Try different chunk sizes and overlap |
| Wrong model for the job | Check MTEB scores — try `bge-large-en-v1.5` |
| LLM not using retrieved context well | Fix your RAG prompt first |
| Users searching within specific categories | Add metadata filters |
| **None of the above + genuinely unusual domain** | **→ Fine-tune** |

### The Domain Distance Test

Fine-tuning helps most when your domain vocabulary is **far from general web text**.

**High distance (fine-tuning likely helps):** IRS correction procedures, clinical trial protocols, semiconductor fab manuals, maritime law

**Low distance (probably not worth it):** customer FAQs, blog posts, general HR policies, product descriptions

> Quick gut-check: if you mixed one of your documents with a random Wikipedia article, would the language difference be jarring? If yes → high domain distance → fine-tuning will help.

---
## 🧪 The Empirical Test

Still not sure? Measure directly. Pick 5–10 query-document pairs where you **know** the document is the correct answer for the query, then compute cosine similarity between query and document embeddings.

| Similarity | What it means |
|---|---|
| > 0.8 | Model already understands your domain — fine-tuning gives marginal gains |
| 0.5–0.8 | Room for improvement — try other fixes first |
| < 0.5 | Model is struggling — fine-tuning is likely worth it |

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 🎛️ Swap the model to compare different base models
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Known relevant query-document pairs from a legal/tax domain
# These are the pairs where you KNOW the document answers the query
queries = [
    "446(b) safe harbor correction method",
    "EPCRS self-correction deadline",
    "plan loan offset rollover rules",
    "section 415 annual additions limit",
    "hardship distribution safe harbor requirements",
]

relevant_docs = [
    "Under Section 446(b), the plan sponsor may use the safe harbor "
    "correction method to fix a missed deferral opportunity by making "
    "a qualified non-elective contribution equal to 50% of the missed deferral.",

    "The Employee Plans Compliance Resolution System allows plan sponsors "
    "to self-correct certain plan failures within three years of the plan "
    "year in which the failure occurred, without IRS approval.",

    "A qualified plan loan offset amount is treated as an eligible rollover "
    "distribution. The participant has until the due date for filing the tax "
    "return for the year of the offset to roll over the amount.",

    "Under IRC Section 415(c), the annual additions limit for defined "
    "contribution plans is the lesser of 100% of compensation or $66,000 "
    "(2023), adjusted for cost-of-living increases.",

    "A hardship distribution from a 401(k) plan satisfies the safe harbor "
    "if it is limited to the employee's elective deferrals and made on "
    "account of an immediate and heavy financial need.",
]

# Encode queries and documents
q_embeddings = model.encode(queries, show_progress_bar=False)
d_embeddings = model.encode(relevant_docs, show_progress_bar=False)

# Compute and display similarity scores
print(f"Model: {model._model_card_text[:40] if hasattr(model, '_model_card_text') else 'BAAI/bge-base-en-v1.5'}")
print(f"{'Query':<45} {'Similarity':>12} {'Verdict'}")
print("-" * 75)

similarities = []
for i in range(len(queries)):
    sim = cosine_similarity([q_embeddings[i]], [d_embeddings[i]])[0][0]
    similarities.append(sim)
    verdict = "✅ Good" if sim > 0.8 else ("⚠️  Okay" if sim > 0.5 else "❌ Weak")
    print(f"{queries[i][:44]:<45} {sim:>12.3f} {verdict}")

print("-" * 75)
avg = np.mean(similarities)
verdict = "Model understands domain — fine-tuning optional" if avg > 0.8 \
          else ("Consider fine-tuning" if avg > 0.5 else "Fine-tuning recommended")
print(f"{'Average':<45} {avg:>12.3f}  → {verdict}")

---
---
## Part 2: Creating Fine-Tuning Data

The quality of your training data matters **more** than any hyperparameter. A small amount of high-quality data beats a large amount of noisy data every time.

You have three ways to get training data — and the best datasets combine all three.

---
### Method 1: Mine From Existing Logs

If you have search logs, click-through data, or human-labelled relevance judgments — that's your gold standard. It reflects **real user behaviour** and **real relevance**.

Sources to look for:
- Search engine click logs (query + clicked document)
- Helpdesk tickets + resolved KB articles
- User upvotes on chatbot answers
- Manually labelled QA pairs

Format them into `(query, positive_document)` pairs and you're ready.

---
### Method 2: LLM-Generated Synthetic Training Data

No logs? Use an LLM to generate training queries directly from your documents.
For each document, the LLM generates realistic questions that the document would answer.

In [ ]:
# ============================================================
# LLM query generation — illustrated with templates
# In production: call your LLM API here
# ============================================================

# Your corpus: the documents you want to make searchable
corpus = [
    "Under Section 446(b), the plan sponsor may use the safe harbor correction "
    "method to fix a missed deferral opportunity by making a qualified "
    "non-elective contribution equal to 50% of the missed deferral.",

    "The Employee Plans Compliance Resolution System allows plan sponsors to "
    "self-correct certain plan failures within three years of the plan year "
    "in which the failure occurred, without IRS approval.",

    "A qualified plan loan offset amount is treated as an eligible rollover "
    "distribution. The participant has until the due date for filing the tax "
    "return for the year of the offset to roll over the amount.",

    "Under IRC Section 415(c), the annual additions limit for defined "
    "contribution plans is the lesser of 100% of compensation or $66,000 "
    "(2023), adjusted for cost-of-living increases.",

    "A hardship distribution from a 401(k) plan satisfies the safe harbor "
    "if it is limited to the employee's elective deferrals and made on "
    "account of an immediate and heavy financial need.",

    "Plan sponsors must provide participants with a summary plan description "
    "within 90 days of becoming covered by the plan. The SPD must describe "
    "the plan's benefits, eligibility requirements, and claims procedures.",

    "The vesting schedule for employer matching contributions must satisfy "
    "either a 3-year cliff vesting schedule or a 2-to-6-year graded "
    "vesting schedule under IRC Section 411(a)(2).",

    "Catch-up contributions allow participants aged 50 or older to contribute "
    "an additional $7,500 (2023) to their 401(k) plan beyond the standard "
    "elective deferral limit of $22,500.",
]

# Prompt template you would send to an LLM
QUERY_GENERATION_PROMPT = """\
You are a retirement plan administrator. Given the following document excerpt,
generate 2-3 realistic search queries that a plan sponsor or HR professional
might type to find this document. Make queries specific and varied.

Document:
{document}

Queries (one per line):"""

# Pre-generated queries (what your LLM would return)
llm_generated_pairs = [
    ("446(b) safe harbor correction for missed deferrals",      corpus[0]),
    ("how to fix missed deferral without IRS approval",         corpus[0]),
    ("EPCRS self-correction period how many years",             corpus[1]),
    ("self-correct plan failure without contacting IRS",        corpus[1]),
    ("plan loan offset rollover deadline tax return",           corpus[2]),
    ("qualified plan loan offset eligible rollover",            corpus[2]),
    ("415 annual additions limit 2023",                         corpus[3]),
    ("defined contribution plan contribution cap",              corpus[3]),
    ("401k hardship distribution safe harbor requirements",     corpus[4]),
    ("immediate and heavy financial need definition",           corpus[4]),
    ("summary plan description deadline 90 days",              corpus[5]),
    ("when must SPD be provided to new participants",           corpus[5]),
    ("employer match vesting schedule cliff graded",            corpus[6]),
    ("411(a)(2) vesting requirements for matching contributions",corpus[6]),
    ("catch-up contribution limit age 50",                      corpus[7]),
    ("how much extra can over-50 employees contribute 401k",    corpus[7]),
]

print(f"✅ Generated {len(llm_generated_pairs)} synthetic query-document pairs")
print(f"\nSample pairs:")
for query, doc in llm_generated_pairs[:4]:
    print(f"  Q: {query}")
    print(f"  D: {doc[:80]}...")
    print()

---
### Method 3: Mine Hard Negatives with BM25

Not all negatives are created equal.

- **Easy negative**: a weather article for a legal query — obviously irrelevant, the model learns nothing
- **Hard negative**: a document that shares keywords with the query but isn't actually relevant — forces the model to learn subtle distinctions

BM25 is perfect for mining hard negatives: it finds keyword-similar documents, exactly the ones an embedding model needs to learn to distinguish from the true positive.

In [ ]:
from rank_bm25 import BM25Okapi

# Index the full corpus with BM25
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def mine_hard_negatives(query, positive_doc, corpus, bm25, n=3):
    """
    Use BM25 to find the top keyword-matching documents that are NOT
    the positive document. These are hard negatives.

    Args:
        query:        the search query
        positive_doc: the known-relevant document (excluded from results)
        corpus:       full list of documents
        bm25:         fitted BM25Okapi index
        n:            number of hard negatives to return

    Returns:
        list of hard negative document strings
    """
    scores = bm25.get_scores(query.lower().split())
    ranked_indices = scores.argsort()[::-1]  # highest BM25 score first

    hard_negs = []
    for idx in ranked_indices:
        candidate = corpus[idx]
        if candidate != positive_doc and len(hard_negs) < n:
            hard_negs.append(candidate)

    return hard_negs


# Build triplets: (query, positive, hard_negative)
triplets = []
for query, positive in llm_generated_pairs:
    hard_negs = mine_hard_negatives(query, positive, corpus, bm25, n=1)
    if hard_negs:
        triplets.append((query, positive, hard_negs[0]))

print(f"✅ Built {len(triplets)} triplets with hard negatives\n")
print("Example triplet:")
print("-" * 65)
q, pos, neg = triplets[0]
print(f"  Query:    {q}")
print(f"  Positive: {pos[:80]}...")
print(f"  Negative: {neg[:80]}...")
print()
print("💡 Notice: the hard negative shares tax/plan keywords with the query")
print("   but is NOT the correct answer. This is what teaches the model")
print("   to go beyond surface-level keyword matching.")

---
---
## Part 3: The Fine-Tuning Process

The `sentence-transformers` library makes this surprisingly concise. The complexity is in the **data preparation** (Part 2) — not the training code.

### The Two Loss Functions

**Triplet Loss** — explicit (anchor, positive, negative) triplets:
```
Loss = max(0, distance(anchor, positive) − distance(anchor, negative) + margin)
```
"Make the positive closer than the negative by at least `margin`. If already satisfied, loss = 0."

**Multiple Negatives Ranking Loss (MNRL)** — pairs only, other batch items become negatives:
- Batch size 32 → 31 free negatives per pair
- More data-efficient, generally better results
- **Default choice for most fine-tuning setups**

---
### Step 1: Prepare Training Examples

In [ ]:
from sentence_transformers import InputExample

# ── MNRL pairs (query + positive document) ───────────────────────────────────
# The other positives in each batch automatically serve as negatives.
# This is the recommended format for most fine-tuning scenarios.
pair_examples = [
    InputExample(texts=[query, positive_doc])
    for query, positive_doc in llm_generated_pairs
]

# ── Triplet examples (query + positive + explicit hard negative) ──────────────
# Use when you've mined high-quality hard negatives and want to be explicit.
triplet_examples = [
    InputExample(texts=[query, positive, hard_neg])
    for query, positive, hard_neg in triplets
]

print(f"✅ Pair examples (for MNRL):    {len(pair_examples)}")
print(f"✅ Triplet examples:             {len(triplet_examples)}")
print(f"\nSample pair example:")
print(f"  texts[0]: {pair_examples[0].texts[0]}")
print(f"  texts[1]: {pair_examples[0].texts[1][:80]}...")
print(f"\nSample triplet example:")
print(f"  texts[0] (query):    {triplet_examples[0].texts[0]}")
print(f"  texts[1] (positive): {triplet_examples[0].texts[1][:70]}...")
print(f"  texts[2] (negative): {triplet_examples[0].texts[2][:70]}...")

---
### Step 2: Set Up Model, DataLoader, and Loss

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

# Start from a pre-trained model — we nudge it toward our domain,
# not overwrite everything it already knows.
# 🎛️ Swap the base model here to experiment with different starting points.
BASE_MODEL = 'BAAI/bge-base-en-v1.5'
ft_model   = SentenceTransformer(BASE_MODEL)

# DataLoader — shuffle to prevent the model from learning order artifacts.
# Larger batch size → more in-batch negatives for MNRL → better training.
# 🎛️ Reduce batch_size if you run out of memory.
train_dataloader = DataLoader(
    pair_examples,
    shuffle=True,
    batch_size=16,
)

# Multiple Negatives Ranking Loss
# Each (query, positive) pair uses all other positives in the batch as negatives.
train_loss = losses.MultipleNegativesRankingLoss(model=ft_model)

print(f"✅ Base model loaded: {BASE_MODEL}")
print(f"   Training examples : {len(pair_examples)}")
print(f"   Batch size        : 16")
print(f"   In-batch negatives: ~15 per pair")
print(f"   Loss function     : MultipleNegativesRankingLoss")

---
### Step 3: Train

With a small dataset like this, training completes in seconds on CPU.

**Hyperparameters that actually matter:**

| Parameter | Rule of thumb |
|---|---|
| `epochs` | 1–5 — more risks catastrophic forgetting |
| `warmup_steps` | ~10% of total steps |
| learning rate | Default (`2e-5`) — lower = safer |
| `batch_size` | As large as memory allows — more in-batch negatives |

> ⚠️ **Catastrophic forgetting**: too many epochs or a high learning rate causes the model to overwrite its general language understanding while specialising on your domain. It gets great at legal docs but suddenly can't handle a simple HR policy question.

In [ ]:
OUTPUT_PATH = "./finetuned-legal-embeddings"

# 🎛️ Tune these — but start with the safe defaults
EPOCHS       = 3
WARMUP_STEPS = 10

ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUTPUT_PATH,
    show_progress_bar=True,
)

print(f"\n✅ Fine-tuned model saved to: {OUTPUT_PATH}")
print(f"   Reload it anytime with:")
print(f"   model = SentenceTransformer('{OUTPUT_PATH}')")

---
---
## Part 4: Did It Actually Work?

Hope is not a measurement strategy. Evaluate on held-out data the model **never saw** during training.

### The Key Metric: Recall@k

For RAG, the most important question is: **"Is the relevant document in the top-k results?"**
If the retriever doesn't find it, the LLM never sees it.

$$\text{Recall@k} = \frac{\text{queries where relevant doc appears in top-}k}{\text{total queries}}$$

In [ ]:
def evaluate_recall_at_k(model, queries, relevant_docs, corpus, k=5):
    """
    Compute Recall@k: the fraction of queries where the known-relevant
    document appears in the top-k retrieved results.

    Args:
        model:         SentenceTransformer model to evaluate
        queries:       list of query strings
        relevant_docs: list of known-relevant documents (one per query)
        corpus:        full document corpus to search over
        k:             how many results to retrieve per query

    Returns:
        recall score between 0.0 and 1.0
    """
    corpus_embeddings = model.encode(corpus,  show_progress_bar=False)
    query_embeddings  = model.encode(queries, show_progress_bar=False)

    hits = 0
    for i, query_emb in enumerate(query_embeddings):
        sims        = cosine_similarity([query_emb], corpus_embeddings)[0]
        top_k_idx   = sims.argsort()[-k:][::-1]
        relevant_idx = corpus.index(relevant_docs[i])
        if relevant_idx in top_k_idx:
            hits += 1

    return hits / len(queries)


# ── Evaluation set (held out — model never saw these during training) ─────────
eval_queries = [
    "how to correct a missed deferral under safe harbor",
    "maximum annual additions defined contribution plan",
    "when must plan summary be given to new employee",
    "employer matching vesting cliff schedule",
    "extra deferral contribution for employees over 50",
    "EPCRS correction without IRS approval timeframe",
    "plan loan offset rollover how long do I have",
    "what qualifies as immediate financial need hardship",
]

eval_relevant = [
    corpus[0],  # 446(b) safe harbor
    corpus[3],  # 415 annual additions
    corpus[5],  # SPD 90-day deadline
    corpus[6],  # vesting schedule
    corpus[7],  # catch-up contributions
    corpus[1],  # EPCRS self-correction
    corpus[2],  # plan loan offset rollover
    corpus[4],  # hardship distribution
]

# ── Compare base vs fine-tuned ────────────────────────────────────────────────
base_model = SentenceTransformer(BASE_MODEL)
ft_model   = SentenceTransformer(OUTPUT_PATH)

for k in [1, 3, 5]:
    base_recall = evaluate_recall_at_k(base_model, eval_queries, eval_relevant, corpus, k=k)
    ft_recall   = evaluate_recall_at_k(ft_model,   eval_queries, eval_relevant, corpus, k=k)
    delta       = ft_recall - base_recall
    arrow       = "⬆️ " if delta > 0 else ("⬇️ " if delta < 0 else "➡️ ")
    print(f"Recall@{k:<2}  Base: {base_recall:.3f}  Fine-tuned: {ft_recall:.3f}  {arrow}{delta:+.3f}")

print()
print("Interpreting delta:")
print("  > +0.10  → Meaningful improvement — fine-tuning helped")
print("  0 to +0.10 → Marginal gain — evaluate cost vs benefit")
print("  Negative → Fine-tuning hurt — check for catastrophic forgetting")

---
### Domain Similarity Test: Before vs After

In [ ]:
# Re-run the empirical test from Part 1 with the fine-tuned model
# to see how similarity scores shifted

base_q = base_model.encode(queries, show_progress_bar=False)
base_d = base_model.encode(relevant_docs, show_progress_bar=False)
ft_q   = ft_model.encode(queries, show_progress_bar=False)
ft_d   = ft_model.encode(relevant_docs, show_progress_bar=False)

print(f"{'Query':<42} {'Base':>7} {'FT':>7} {'Δ':>7}")
print("-" * 65)

for i in range(len(queries)):
    base_sim = cosine_similarity([base_q[i]], [base_d[i]])[0][0]
    ft_sim   = cosine_similarity([ft_q[i]],   [ft_d[i]])[0][0]
    delta    = ft_sim - base_sim
    arrow    = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "→")
    print(f"{queries[i][:41]:<42} {base_sim:>7.3f} {ft_sim:>7.3f} {arrow}{delta:>+6.3f}")

base_avg = np.mean([cosine_similarity([base_q[i]], [base_d[i]])[0][0] for i in range(len(queries))])
ft_avg   = np.mean([cosine_similarity([ft_q[i]],   [ft_d[i]])[0][0]   for i in range(len(queries))])
print("-" * 65)
print(f"{'Average':<42} {base_avg:>7.3f} {ft_avg:>7.3f} {ft_avg-base_avg:>+7.3f}")

---
### The BEIR Sanity Check

Fine-tuning can improve your domain **at the cost of general performance**. Your model gets great at legal documents but suddenly struggles with a simple holiday policy question.

BEIR (Benchmarking IR) tests across 18 diverse datasets. You don't need to run all of them — pick 2–3 **outside your domain** as a sanity check.

> **Rule of thumb:** If scores on out-of-domain benchmarks drop more than 5–10%, your fine-tuning is too aggressive. Reduce epochs or learning rate.

In [ ]:
# ── Simulate a BEIR-style out-of-domain check ─────────────────────────────
# In production: pip install beir → run actual BEIR datasets
# Here we simulate with out-of-domain pairs your model should still handle.

ood_queries = [
    "What causes type 2 diabetes?",
    "How does photosynthesis work?",
    "What is the capital of France?",
    "How do I apply for a patent?",
    "What are the symptoms of a concussion?",
]

ood_positives = [
    "Type 2 diabetes occurs when the body becomes resistant to insulin or "
    "doesn't produce enough insulin, leading to elevated blood glucose levels.",

    "Photosynthesis is the process by which plants use sunlight, water, and "
    "carbon dioxide to produce oxygen and energy in the form of glucose.",

    "Paris is the capital and most populous city of France, located in the "
    "north-central part of the country along the Seine River.",

    "A patent application must include a written description of the invention, "
    "claims defining the scope of protection, and an abstract.",

    "Concussion symptoms include headache, confusion, dizziness, memory loss, "
    "nausea, and sensitivity to light. Symptoms usually resolve within days to weeks.",
]

ood_corpus = ood_positives  # small corpus for the demo

base_ood_recall = evaluate_recall_at_k(base_model, ood_queries, ood_positives, ood_corpus, k=1)
ft_ood_recall   = evaluate_recall_at_k(ft_model,   ood_queries, ood_positives, ood_corpus, k=1)

drop = ft_ood_recall - base_ood_recall
print("Out-of-domain (BEIR-style) sanity check:")
print(f"  Base model Recall@1:       {base_ood_recall:.3f}")
print(f"  Fine-tuned model Recall@1: {ft_ood_recall:.3f}")
print(f"  Drop:                      {drop:+.3f}")
print()
if drop < -0.10:
    print("⚠️  Drop > 10% — signs of catastrophic forgetting.")
    print("   Try: fewer epochs, lower learning rate, or adapter-based fine-tuning.")
elif drop < 0:
    print("⚠️  Small drop — monitor in production. May be acceptable.")
else:
    print("✅ No significant out-of-domain degradation.")

---
---
## Part 5: The Decision Framework

```
Results not good enough?
  │
  ├── Fix chunking first (sizes, overlap, strategy)
  │
  ├── Try a larger/domain-adjacent base model (check MTEB)
  │
  ├── Fix the RAG prompt / generation step
  │
  ├── Add metadata filters
  │
  └── STILL not good enough?
        │
        ├── Run empirical domain distance test
        │     ├── avg similarity > 0.8 → don't fine-tune
        │     └── avg similarity < 0.8 → fine-tune candidate
        │
        └── Fine-tune
              ├── Build training data (logs + LLM-generated + BM25 hard negatives)
              ├── Train with MNRL loss (3 epochs, default LR)
              ├── Evaluate Recall@k on held-out set
              └── BEIR sanity check — confirm no catastrophic forgetting
```

---
## 🧪 Exercises

1. **Swap the base model**: Change `BASE_MODEL` to `'all-MiniLM-L6-v2'`. Run the empirical test — does it score higher or lower than `bge-base-en-v1.5` on the legal domain? What does that tell you about which model to use as a fine-tuning starting point?

2. **Catastrophic forgetting**: Set `EPOCHS = 10`. Re-run training and re-run the BEIR sanity check. Does out-of-domain recall drop?

3. **Hard negatives matter**: Remove hard negatives and train using only pair examples. Compare Recall@k. What changed?

4. **Your own domain**: Replace `corpus`, `queries`, and `relevant_docs` with 5–10 documents from your own domain. Run the empirical domain distance test — does your domain warrant fine-tuning?